# Separación Ciega de Fuentes de Vibración

**Artículo**: Blind Separation of Vibration Sources using Deep Learning and Deconvolution

**ArXiv**: [2405.12774](https://arxiv.org/abs/2405.12774)

Este notebook reproduce el método de dos etapas para separar vibraciones de engranaje y rodamiento.

## 1. Preparación: Importaciones y generación de datos

In [ ]:
import matplotlib
import numpy as np

matplotlib.use("Agg")
import sys

import matplotlib.pyplot as plt

sys.path.insert(0, ".")

from data.generate_synthetic_vibration import generate_synthetic_vibration
from src import VibrationSeparator

print("✓ Librerías cargadas correctamente")

## 2. Generación de datos sintéticos

In [ ]:
time, gear_pure, bearing_pure, mixed_noisy, transfer_func = generate_synthetic_vibration(
    sampling_rate=10000,
    duration=2.0,
    freq_gear=100.0,
    freq_bearing=60.0,
    gear_amplitude=1.0,
    bearing_amplitude=0.1,
    snr_db=20.0,
)

print("Datos generados:")
print(f"  - Duración: {time[-1]:.2f} segundos")
print(f"  - Muestras: {len(time)}")
print(f"  - Energía engranaje: {np.var(gear_pure):.4f}")
print(f"  - Energía rodamiento: {np.var(bearing_pure):.4f}")
print(f"  - Energía señal mezclada: {np.var(mixed_noisy):.4f}")

## 3. Inicialización del método de separación

In [ ]:
separator = VibrationSeparator(fft_size=1024, num_cnn_layers=4)

print("✓ Separador inicializado con:")
print(f"  - FFT size: {separator.fft_size}")
print(f"  - Capas CNN: {separator.cnn.num_layers}")

## 4. Separación de fuentes (método con CNN)

In [ ]:
result_cnn = separator.separate_sources(mixed_noisy, use_cnn_for_gear=True)

gear_estimated_cnn = result_cnn["gear"]
bearing_estimated_cnn = result_cnn["bearing"]

print("Separación completada (usando CNN)")
print(
    f"  - Engranaje estimado: min={gear_estimated_cnn.min():.4f}, max={gear_estimated_cnn.max():.4f}"
)
print(
    f"  - Rodamiento estimado: min={bearing_estimated_cnn.min():.4f}, max={bearing_estimated_cnn.max():.4f}"
)

## 5. Separación alternativa (sin CNN, usando envolvente)

In [ ]:
result_envelope = separator.separate_sources(mixed_noisy, use_cnn_for_gear=False)

gear_estimated_env = result_envelope["gear"]
bearing_estimated_env = result_envelope["bearing"]

print("Separación completada (usando envolvente)")

## 6. Visualización: Señal original vs. separada (CNN)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 10))

plot_range = slice(0, min(2000, len(time)))

axes[0].plot(
    time[plot_range], mixed_noisy[plot_range], "gray", alpha=0.7, label="Señal mezclada (observada)"
)
axes[0].set_ylabel("Amplitud (V)")
axes[0].set_title("Señal Mezclada Observada")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(time[plot_range], gear_pure[plot_range], "b-", alpha=0.5, label="Engranaje (real)")
axes[1].plot(
    time[plot_range],
    gear_estimated_cnn[plot_range],
    "r--",
    alpha=0.7,
    label="Engranaje (estimado CNN)",
)
axes[1].set_ylabel("Amplitud (V)")
axes[1].set_title("Vibración de Engranaje: Real vs. Estimada")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(time[plot_range], bearing_pure[plot_range], "g-", alpha=0.5, label="Rodamiento (real)")
axes[2].plot(
    time[plot_range],
    bearing_estimated_cnn[plot_range],
    "orange",
    linestyle="--",
    alpha=0.7,
    label="Rodamiento (estimado)",
)
axes[2].set_ylabel("Amplitud (V)")
axes[2].set_title("Vibración de Rodamiento: Real vs. Estimada")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

residue = mixed_noisy - gear_estimated_cnn
axes[3].plot(
    time[plot_range], residue[plot_range], "purple", alpha=0.7, label="Residuo (después separación)"
)
axes[3].set_xlabel("Tiempo (s)")
axes[3].set_ylabel("Amplitud (V)")
axes[3].set_title("Residuo (Señal - Engranaje Estimado)")
axes[3].legend()
axes[3].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("separation_result.png", dpi=100, bbox_inches="tight")
print("✓ Gráfica guardada: separation_result.png")
plt.show()

## 7. Análisis espectral (antes/después)

In [ ]:
def compute_spectrum(signal, sampling_rate=10000):
    fft_vals = np.fft.fft(signal)
    freqs = np.fft.fftfreq(len(signal), 1 / sampling_rate)
    magnitude = np.abs(fft_vals)
    return freqs[: len(freqs) // 2], magnitude[: len(magnitude) // 2]


freqs_mixed, mag_mixed = compute_spectrum(mixed_noisy)
freqs_gear_est, mag_gear_est = compute_spectrum(gear_estimated_cnn)
freqs_bearing_est, mag_bearing_est = compute_spectrum(bearing_estimated_cnn)

fig, axes = plt.subplots(3, 1, figsize=(12, 9))

freq_range = slice(0, np.argmax(freqs_mixed > 500))

axes[0].semilogy(freqs_mixed[freq_range], mag_mixed[freq_range], "gray", linewidth=0.5)
axes[0].set_ylabel("Magnitud (V)")
axes[0].set_title("Espectro: Señal Mezclada")
axes[0].grid(True, alpha=0.3, which="both")

axes[1].semilogy(freqs_gear_est[freq_range], mag_gear_est[freq_range], "r", linewidth=0.7)
axes[1].axvline(
    100, color="b", linestyle="--", alpha=0.5, label="Freq. engranaje esperada (100 Hz)"
)
axes[1].set_ylabel("Magnitud (V)")
axes[1].set_title("Espectro: Vibración de Engranaje Estimada")
axes[1].legend()
axes[1].grid(True, alpha=0.3, which="both")

axes[2].semilogy(
    freqs_bearing_est[freq_range], mag_bearing_est[freq_range], "orange", linewidth=0.7
)
axes[2].axvline(60, color="g", linestyle="--", alpha=0.5, label="Freq. rodamiento esperada (60 Hz)")
axes[2].set_xlabel("Frecuencia (Hz)")
axes[2].set_ylabel("Magnitud (V)")
axes[2].set_title("Espectro: Vibración de Rodamiento Estimada")
axes[2].legend()
axes[2].grid(True, alpha=0.3, which="both")

plt.tight_layout()
plt.savefig("spectral_analysis.png", dpi=100, bbox_inches="tight")
print("✓ Gráfica guardada: spectral_analysis.png")
plt.show()

## 8. Comparación de métodos: CNN vs. Envolvente

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

plot_range = slice(0, min(1000, len(time)))

ax.plot(
    time[plot_range],
    bearing_pure[plot_range],
    "g-",
    alpha=0.6,
    linewidth=2,
    label="Real (ground truth)",
)
ax.plot(
    time[plot_range],
    bearing_estimated_cnn[plot_range],
    "r--",
    alpha=0.7,
    linewidth=1.5,
    label="Estimado (CNN)",
)
ax.plot(
    time[plot_range],
    bearing_estimated_env[plot_range],
    "b:",
    alpha=0.7,
    linewidth=1.5,
    label="Estimado (Envolvente)",
)

ax.set_xlabel("Tiempo (s)", fontsize=11)
ax.set_ylabel("Amplitud (V)", fontsize=11)
ax.set_title("Comparación de Métodos: CNN vs. Envolvente", fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("method_comparison.png", dpi=100, bbox_inches="tight")
print("✓ Gráfica guardada: method_comparison.png")
plt.show()

## 9. Métricas de desempeño

In [ ]:
def compute_mse(predicted, true):
    return np.mean((predicted - true) ** 2)


def compute_correlation(predicted, true):
    return np.corrcoef(predicted, true)[0, 1]


mse_gear = compute_mse(gear_estimated_cnn, gear_pure)
mse_bearing = compute_mse(bearing_estimated_cnn, bearing_pure)

corr_gear = compute_correlation(gear_estimated_cnn, gear_pure)
corr_bearing = compute_correlation(bearing_estimated_cnn, bearing_pure)

print("=" * 60)
print("MÉTRICAS DE DESEMPEÑO (Método CNN)")
print("=" * 60)
print("\n🔧 VIBRACIÓN DE ENGRANAJE:")
print(f"   MSE: {mse_gear:.6f}")
print(f"   Correlación: {corr_gear:.6f}")
print("\n⚙️  VIBRACIÓN DE RODAMIENTO:")
print(f"   MSE: {mse_bearing:.6f}")
print(f"   Correlación: {corr_bearing:.6f}")
print("\n📊 SUMARIO:")
print("   Separación completada exitosamente")
print(f"   - Energía engranaje real: {np.var(gear_pure):.4f}")
print(f"   - Energía engranaje estimado: {np.var(gear_estimated_cnn):.4f}")
print(f"   - Energía rodamiento real: {np.var(bearing_pure):.4f}")
print(f"   - Energía rodamiento estimado: {np.var(bearing_estimated_cnn):.4f}")
print("=" * 60)

## 10. Conclusión

El método propuesto en Makienko et al. logra separar exitosamente las fuentes de vibración sin información previa del sistema. Las características espectrales y la envolvente se preservan correctamente, permitiendo la detección temprana de anomalías en rodamientos.